**ANÁLISE SENTIMENTO - PARTE 2 DO CÓDIGO**

In [0]:
%sql
-- ============================================================
-- Validação 6: contagem por sentimento consolidado (Terra Signal, history)
-- ============================================================
-- O que faz: conta as linhas (cliente x aspecto) por Sentimento_Consolidado, com o percentual do total.
-- Origem:    terra_signal_history_aspects
-- Saída:     consulta (Sentimento_Consolidado, linhas, pct_total), sem criar tabela.
-- Regras:    o percentual é sobre o total de linhas da tabela.
-- ============================================================

SELECT
  Sentimento_Consolidado,
  COUNT(*) AS linhas,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_total
FROM hackaton_databricks.tabelas_hackaton.terra_signal_history_aspects
GROUP BY Sentimento_Consolidado
ORDER BY linhas DESC;

Sentimento_Consolidado,linhas,pct_total
Sem avaliacao,9384,49.2
Positivo,9006,47.2
Negativo,515,2.7
Misto,169,0.9


In [0]:
%sql
-- ============================================================
-- Tabela terra_signal_feedback_aspects (Terra Signal, inference)
-- ============================================================
-- O que faz: carrega o arquivo CSV da Terra Signal (aspectos, inference) em tabela.
-- Origem:    terra_signal_feedback_aspects.csv (volume bases, schema bases)
-- Saída:     tabela terra_signal_feedback_aspects, com as colunas do CSV, todas como texto.
-- Regras:    separador do CSV ";" e cabeçalho na primeira linha. Os tipos não são inferidos.
-- ============================================================

CREATE OR REPLACE TABLE hackaton_databricks.tabelas_hackaton.terra_signal_feedback_aspects AS
SELECT *
FROM read_files(
  '/Volumes/hackaton_databricks/bases/bases/terra_signal_feedback_aspects.csv',
  format => 'csv',
  sep => ';',
  header => true,
  inferColumnTypes => false
);

SELECT COUNT(*) AS linhas
FROM hackaton_databricks.tabelas_hackaton.terra_signal_feedback_aspects;

SELECT *
FROM hackaton_databricks.tabelas_hackaton.terra_signal_feedback_aspects
LIMIT ALL;

customerID,Aspecto,Sentimento_Consolidado,Qtd_Mencoes,Qtd_Positivas,Qtd_Negativas,Qtd_Mistas,Evidencia_Mista,Evidencia_Negativa,Evidencia_Positiva,_rescued_data
0004-TLHLJ,Cobranca,Negativo,1,0,1,0,null,the payment method of electronic check was not convenient for me.,null,null
0004-TLHLJ,Conexao,Sem avaliacao,2,0,0,0,null,null,null,null
0004-TLHLJ,Contrato,Sem avaliacao,1,0,0,0,null,null,null,null
0004-TLHLJ,Preco,Sem avaliacao,1,0,0,0,null,null,null,null
0013-EXCHZ,Cobranca,Negativo,1,0,1,0,null,but I found the process of sending a mailed check for payment to be inconvenient.,null,null
0013-EXCHZ,Conexao,Sem avaliacao,1,0,0,0,null,null,null,null
0013-EXCHZ,Preco,Positivo,1,1,0,0,null,null,The monthly charges of $83.9 for fiber optic internet were reasonable,null
0020-JDNXP,Cobranca,Positivo,1,1,0,0,null,null,I appreciate being able to pay via mailed check as it is convenient for me.,null
0020-JDNXP,Conexao,Sem avaliacao,2,0,0,0,null,null,null,null
0020-JDNXP,Preco,Positivo,1,1,0,0,null,null,The monthly charges of $61.25 are reasonable for the speed and reliability of the connection.,null


In [0]:
%sql
-- ============================================================
-- Validação 7: contagem por sentimento consolidado (Terra Signal, inference)
-- ============================================================
-- O que faz: conta as linhas (cliente x aspecto) por Sentimento_Consolidado, com o percentual do total.
-- Origem:    terra_signal_feedback_aspects
-- Saída:     consulta (Sentimento_Consolidado, linhas, pct_total), sem criar tabela.
-- Regras:    o percentual é sobre o total de linhas da tabela.
-- ============================================================

SELECT
  Sentimento_Consolidado,
  COUNT(*) AS linhas,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_total
FROM hackaton_databricks.tabelas_hackaton.terra_signal_feedback_aspects
GROUP BY Sentimento_Consolidado
ORDER BY linhas DESC;

Sentimento_Consolidado,linhas,pct_total
Sem avaliacao,2325,48.8
Positivo,2278,47.8
Negativo,131,2.7
Misto,35,0.7


In [0]:
%sql
-- ============================================================
-- Validação 8: clientes do consolidado x Terra Signal (history)
-- ============================================================
-- O que faz: confere se os clientes de feedback_consolidado_history estão em terra_signal_history_aspects, e o contrário.
-- Origem:    feedback_consolidado_history e terra_signal_history_aspects
-- Saída:     consulta (clientes em cada tabela, clientes só no consolidado, clientes só na Terra Signal), sem criar tabela.
-- Regras:    compara só customerID, sem olhar Aspecto.
-- ============================================================

WITH a AS (
  SELECT DISTINCT customerID
  FROM hackaton_databricks.tabelas_hackaton.feedback_consolidado_history
),
b AS (
  SELECT DISTINCT customerID
  FROM hackaton_databricks.tabelas_hackaton.terra_signal_history_aspects
)
SELECT
  (SELECT COUNT(*) FROM a) AS clientes_consolidado,
  (SELECT COUNT(*) FROM b) AS clientes_terra_signal,
  (SELECT COUNT(*) FROM a LEFT JOIN b ON a.customerID = b.customerID
    WHERE b.customerID IS NULL) AS so_no_consolidado,
  (SELECT COUNT(*) FROM b LEFT JOIN a ON a.customerID = b.customerID
    WHERE a.customerID IS NULL) AS so_na_terra_signal;

clientes_consolidado,clientes_terra_signal,so_no_consolidado,so_na_terra_signal
5634,5634,0,0


In [0]:
%sql
-- ============================================================
-- Validação 9: sentimento do consolidado x Terra Signal (history)
-- ============================================================
-- O que faz: para cada sentimento do consolidado, mostra quantas linhas (cliente x aspecto) têm o mesmo sentimento na Terra Signal e quantas foram para cada um dos outros.
-- Origem:    feedback_consolidado_history e terra_signal_history_aspects
-- Saída:     consulta, 4 linhas (uma por sentimento do consolidado), com total e contagem por sentimento na Terra Signal, sem criar tabela.
-- Regras:    base = consolidado, com LEFT JOIN por customerID e Aspecto. A coluna com o mesmo nome da linha é a que coincide; as demais são as que divergem. nao_encontrado = cliente e aspecto ausentes na Terra Signal. outro_valor = sentimento da Terra Signal com escrita diferente das quatro esperadas. Linhas que existem só na Terra Signal não entram.
-- ============================================================

SELECT
  c.Sentimento_Consolidado AS sentimento_consolidado,
  COUNT(*) AS total,
  SUM(CASE WHEN t.Sentimento_Consolidado = 'Misto' THEN 1 ELSE 0 END) AS na_terra_misto,
  SUM(CASE WHEN t.Sentimento_Consolidado = 'Positivo' THEN 1 ELSE 0 END) AS na_terra_positivo,
  SUM(CASE WHEN t.Sentimento_Consolidado = 'Negativo' THEN 1 ELSE 0 END) AS na_terra_negativo,
  SUM(CASE WHEN t.Sentimento_Consolidado = 'Sem avaliacao' THEN 1 ELSE 0 END) AS na_terra_sem_avaliacao,
  SUM(CASE WHEN t.customerID IS NULL THEN 1 ELSE 0 END) AS nao_encontrado,
  SUM(CASE WHEN t.customerID IS NOT NULL
            AND t.Sentimento_Consolidado NOT IN ('Misto','Positivo','Negativo','Sem avaliacao')
           THEN 1 ELSE 0 END) AS outro_valor
FROM hackaton_databricks.tabelas_hackaton.feedback_consolidado_history c
LEFT JOIN hackaton_databricks.tabelas_hackaton.terra_signal_history_aspects t
  ON c.customerID = t.customerID AND c.Aspecto = t.Aspecto
GROUP BY c.Sentimento_Consolidado
ORDER BY c.Sentimento_Consolidado;

sentimento_consolidado,total,na_terra_misto,na_terra_positivo,na_terra_negativo,na_terra_sem_avaliacao,nao_encontrado,outro_valor
Misto,3308,164,1743,250,1124,27,0
Negativo,2436,4,11,263,597,1561,0
Positivo,15482,1,7252,2,5637,2590,0
Sem avaliacao,2063,0,0,0,2025,38,0


In [0]:
%sql
-- ============================================================
-- Validação 10: aspectos do consolidado x Terra Signal (history)
-- ============================================================
-- O que faz: lista os aspectos de cada tabela, com a quantidade de linhas, para achar nomes que existem só em uma delas.
-- Origem:    feedback_consolidado_history e terra_signal_history_aspects
-- Saída:     consulta (aspecto, linhas_consolidado, linhas_terra_signal), um aspecto por linha, sem criar tabela.
-- Regras:    FULL OUTER JOIN pelo nome do Aspecto. Null em um dos lados indica aspecto que só existe na outra tabela.
-- ============================================================

WITH a AS (
  SELECT Aspecto, COUNT(*) AS linhas
  FROM hackaton_databricks.tabelas_hackaton.feedback_consolidado_history
  GROUP BY Aspecto
),
b AS (
  SELECT Aspecto, COUNT(*) AS linhas
  FROM hackaton_databricks.tabelas_hackaton.terra_signal_history_aspects
  GROUP BY Aspecto
)
SELECT
  COALESCE(a.Aspecto, b.Aspecto) AS aspecto,
  a.linhas AS linhas_consolidado,
  b.linhas AS linhas_terra_signal
FROM a
FULL OUTER JOIN b ON a.Aspecto = b.Aspecto
ORDER BY aspecto;

aspecto,linhas_consolidado,linhas_terra_signal
Atendimento,160,160
Cobranca,5599,5402
Conexao,5621,5619
Contrato,2370,2369
Preco,5610,5523
Saida,3841,null
Seguranca,1,null
Streaming,87,null
Suporte,null,1


In [0]:
%sql
-- ============================================================
-- Validação 11: aspectos do consolidado x Terra Signal (inference)
-- ============================================================
-- O que faz: lista os aspectos de cada tabela, com a quantidade de linhas, para achar nomes que existem só em uma delas.
-- Origem:    feedback_consolidado_inference e terra_signal_feedback_aspects
-- Saída:     consulta (aspecto, linhas_consolidado_inference, linhas_terra_signal_feedback), um aspecto por linha, sem criar tabela.
-- Regras:    FULL OUTER JOIN pelo nome do Aspecto. Null em um dos lados indica aspecto que só existe na outra tabela.
-- ============================================================

WITH a AS (
  SELECT Aspecto, COUNT(*) AS linhas
  FROM hackaton_databricks.tabelas_hackaton.feedback_consolidado_inference
  GROUP BY Aspecto
),
b AS (
  SELECT Aspecto, COUNT(*) AS linhas
  FROM hackaton_databricks.tabelas_hackaton.terra_signal_feedback_aspects
  GROUP BY Aspecto
)
SELECT
  COALESCE(a.Aspecto, b.Aspecto) AS aspecto,
  a.linhas AS linhas_consolidado_inference,
  b.linhas AS linhas_terra_signal_feedback
FROM a
FULL OUTER JOIN b ON a.Aspecto = b.Aspecto
ORDER BY aspecto;

aspecto,linhas_consolidado_inference,linhas_terra_signal_feedback
Atendimento,32,32
Cobranca,1393,1339
Conexao,1407,1407
Contrato,610,610
Preco,1400,1381
Saida,986,null
Streaming,24,null


In [0]:
%sql
-- ============================================================
-- Validação 12: linhas por sentimento, consolidado x Terra Signal (history)
-- ============================================================
-- O que faz: compara quantas linhas cada tabela tem por Sentimento_Consolidado.
-- Origem:    feedback_consolidado_history e terra_signal_history_aspects
-- Saída:     consulta (sentimento, linhas_consolidado, linhas_terra_signal, diferenca), uma linha por sentimento, sem criar tabela.
-- Regras:    só contagem por sentimento, sem ligar cliente e aspecto. diferenca = consolidado menos Terra Signal.
-- ============================================================

WITH a AS (
  SELECT Sentimento_Consolidado AS sentimento, COUNT(*) AS linhas
  FROM hackaton_databricks.tabelas_hackaton.feedback_consolidado_history
  GROUP BY 1
),
b AS (
  SELECT Sentimento_Consolidado AS sentimento, COUNT(*) AS linhas
  FROM hackaton_databricks.tabelas_hackaton.terra_signal_history_aspects
  GROUP BY 1
)
SELECT
  COALESCE(a.sentimento, b.sentimento) AS sentimento,
  a.linhas AS linhas_consolidado,
  b.linhas AS linhas_terra_signal,
  COALESCE(a.linhas, 0) - COALESCE(b.linhas, 0) AS diferenca
FROM a
FULL OUTER JOIN b ON a.sentimento = b.sentimento
ORDER BY sentimento;

sentimento,linhas_consolidado,linhas_terra_signal,diferenca
Misto,3308,169,3139
Negativo,2436,515,1921
Positivo,15482,9006,6476
Sem avaliacao,2063,9384,-7321


In [0]:
%sql
-- ============================================================
-- Validação 12: linhas por sentimento, consolidado x Terra Signal (history)
-- ============================================================
-- O que faz: compara quantas linhas cada tabela tem por Sentimento_Consolidado.
-- Origem:    feedback_consolidado_history e terra_signal_history_aspects
-- Saída:     consulta (sentimento, linhas_consolidado, linhas_terra_signal, diferenca), uma linha por sentimento, sem criar tabela.
-- Regras:    só contagem por sentimento, sem ligar cliente e aspecto. diferenca = consolidado menos Terra Signal.
-- ============================================================

WITH a AS (
  SELECT Sentimento_Consolidado AS sentimento, COUNT(*) AS linhas
  FROM hackaton_databricks.tabelas_hackaton.feedback_consolidado_history
  GROUP BY 1
),
b AS (
  SELECT Sentimento_Consolidado AS sentimento, COUNT(*) AS linhas
  FROM hackaton_databricks.tabelas_hackaton.terra_signal_history_aspects
  GROUP BY 1
)
SELECT
  COALESCE(a.sentimento, b.sentimento) AS sentimento,
  a.linhas AS linhas_consolidado,
  b.linhas AS linhas_terra_signal,
  COALESCE(a.linhas, 0) - COALESCE(b.linhas, 0) AS diferenca
FROM a
FULL OUTER JOIN b ON a.sentimento = b.sentimento
ORDER BY sentimento;

sentimento,linhas_consolidado,linhas_terra_signal,diferenca
Misto,872,35,837
Negativo,603,131,472
Positivo,3855,2278,1577
Sem avaliacao,522,2325,-1803


In [0]:
%python
# ============================================================
# Sentimento global e VADER por cliente (history)
# ============================================================
# O que faz: gera uma linha por cliente, com os dados da base, as contagens de sentimento por aspecto, o SentimentoGlobal e o sentimento do texto do feedback (VADER).
# Origem:    history_tipada e feedback_consolidado_history
# Saída:     tabela terra_signal_history_intelligence_Adri (colunas da history_tipada, QtdAspectosAvaliados, QtdPositivos, QtdNegativos, QtdMistos, SentimentoGlobal, SentimentScore, SentimentoVADER, Base).
# Regras:    Saida entra nas contagens e no SentimentoGlobal. QtdAspectosAvaliados não conta Sem avaliacao. QtdPositivos e QtdNegativos somam menções. QtdMistos é 1 se algum aspecto tem menção mista. SentimentoGlobal é Misto se há aspecto Misto ou Positivo e Negativo juntos; senão Positivo, Negativo ou Sem avaliacao. SentimentScore é o compound do VADER (vaderSentiment) sobre o texto completo, com 4 casas; SentimentoVADER é Positivo (>= 0,05), Negativo (<= -0,05) ou Neutro. Base recebe "History".
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf
import pandas as pd

@pandas_udf("double")
def vader(s: pd.Series) -> pd.Series:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    a = SentimentIntensityAnalyzer()
    return s.map(lambda t: float(a.polarity_scores(t)["compound"]) if t else None)

base = spark.sql("""
SELECT
  h.customerID, h.gender, h.SeniorCitizen, h.Partner, h.Dependents, h.tenure,
  h.PhoneService, h.MultipleLines, h.InternetService, h.OnlineSecurity, h.OnlineBackup,
  h.DeviceProtection, h.TechSupport, h.StreamingTV, h.StreamingMovies, h.Contract,
  h.PaperlessBilling, h.PaymentMethod, h.MonthlyCharges, h.TotalCharges, h.Churn,
  h.CustomerFeedback, h.MonthlyIncome,
  a.QtdAspectosAvaliados, a.QtdPositivos, a.QtdNegativos, a.QtdMistos, a.SentimentoGlobal,
  'History' AS Base
FROM hackaton_databricks.tabelas_hackaton.history_tipada h
JOIN (
  SELECT
    customerID,
    SUM(CASE WHEN Sentimento_Consolidado <> 'Sem avaliacao' THEN 1 ELSE 0 END) AS QtdAspectosAvaliados,
    SUM(Qtd_Positivas) AS QtdPositivos,
    SUM(Qtd_Negativas) AS QtdNegativos,
    MAX(Qtd_Mistas) AS QtdMistos,
    CASE
      WHEN SUM(CASE WHEN Sentimento_Consolidado = 'Misto' THEN 1 ELSE 0 END) > 0 THEN 'Misto'
      WHEN SUM(CASE WHEN Sentimento_Consolidado = 'Positivo' THEN 1 ELSE 0 END) > 0
       AND SUM(CASE WHEN Sentimento_Consolidado = 'Negativo' THEN 1 ELSE 0 END) > 0 THEN 'Misto'
      WHEN SUM(CASE WHEN Sentimento_Consolidado = 'Positivo' THEN 1 ELSE 0 END) > 0 THEN 'Positivo'
      WHEN SUM(CASE WHEN Sentimento_Consolidado = 'Negativo' THEN 1 ELSE 0 END) > 0 THEN 'Negativo'
      ELSE 'Sem avaliacao'
    END AS SentimentoGlobal
  FROM hackaton_databricks.tabelas_hackaton.feedback_consolidado_history
  GROUP BY customerID
) a ON h.customerID = a.customerID
""")

res = (base
       .withColumn("SentimentScore", F.round(vader("CustomerFeedback"), 4))
       .withColumn("SentimentoVADER",
                   F.when(F.col("SentimentScore") >= 0.05, "Positivo")
                    .when(F.col("SentimentScore") <= -0.05, "Negativo")
                    .otherwise("Neutro")))

colunas = [c for c in base.columns if c != "Base"] + ["SentimentScore", "SentimentoVADER", "Base"]

(res.select(colunas)
    .write.mode("overwrite")
    .saveAsTable("hackaton_databricks.tabelas_hackaton.terra_signal_history_intelligence_Adri"))

spark.sql("""
SELECT COUNT(*) AS linhas, COUNT(DISTINCT customerID) AS clientes
FROM hackaton_databricks.tabelas_hackaton.terra_signal_history_intelligence_Adri
""").show()

spark.sql("""
SELECT customerID, QtdAspectosAvaliados, QtdPositivos, QtdNegativos, QtdMistos,
       SentimentoGlobal, SentimentScore, SentimentoVADER, Base
FROM hackaton_databricks.tabelas_hackaton.terra_signal_history_intelligence_Adri
WHERE customerID = '0578-SKVMF'
""").show(truncate=False)

+------+--------+
|linhas|clientes|
+------+--------+
|  5634|    5634|
+------+--------+

+----------+--------------------+------------+------------+---------+----------------+--------------+---------------+-------+
|customerID|QtdAspectosAvaliados|QtdPositivos|QtdNegativos|QtdMistos|SentimentoGlobal|SentimentScore|SentimentoVADER|Base   |
+----------+--------------------+------------+------------+---------+----------------+--------------+---------------+-------+
|0578-SKVMF|4                   |2           |3           |1        |Misto           |0.2023        |Positivo       |History|
+----------+--------------------+------------+------------+---------+----------------+--------------+---------------+-------+



In [0]:
%sql
-- ============================================================
-- Visualização e download da tabela (history)
-- ============================================================
-- O que faz: exibe a tabela terra_signal_history_intelligence_Adri, uma linha por cliente, para consulta e download em .csv.
-- Origem:    terra_signal_history_intelligence_Adri
-- Saída:     consulta com todas as colunas, ordenada por customerID, sem criar tabela.
-- Regras:    sem limite de linhas. O download direto pela tela trunca o resultado (exibe só parte das linhas); para a base completa, usar a exportação para volume.
-- ============================================================

SELECT *
FROM hackaton_databricks.tabelas_hackaton.terra_signal_history_intelligence_Adri
ORDER BY customerID
LIMIT ALL;

customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,CustomerFeedback,MonthlyIncome,QtdAspectosAvaliados,QtdPositivos,QtdNegativos,QtdMistos,SentimentoGlobal,SentimentScore,SentimentoVADER,Base
0002-ORFBO,Female,0,Yes,Yes,9,Yes,No,DSL,No,Yes,No,Yes,Yes,No,One year,Yes,Mailed check,65.60,593.30,No,"I have been a customer with this company for 9 months now and I am pleased to say that I have not experienced any issues with my service. The DSL internet service has been reliable and I have not had any disruptions in my connection. I appreciate the affordable monthly charge of $65.6 and the option to pay by mailed check. Overall, I am satisfied with my experience and have no plans to churn.",3879,4,4,0,0,Positivo,0.7805,Positivo,History
0003-MKNFE,Male,0,No,No,9,Yes,Yes,DSL,No,No,No,No,No,Yes,Month-to-month,No,Mailed check,59.90,542.40,No,"I have been a customer with this internet service provider for 9 months now and I am very satisfied with the service. The DSL internet has been reliable and I have not experienced any issues with connectivity. The monthly charges of $59.90 are reasonable for the service provided. I appreciate the option to pay by mailed check as it is convenient for me. Overall, I have no complaints and will continue to use this service.",7701,3,4,0,0,Positivo,0.6758,Positivo,History
0011-IGKFF,Male,1,Yes,No,13,Yes,No,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,98.00,1237.85,Yes,"I have been a customer with this company for 13 months now and unfortunately, I have decided to churn. While the fiber optic internet service was fast and reliable, I found the monthly charges of $98 to be quite high. Additionally, the payment method of electronic check was convenient, but I was hoping for more competitive pricing options and possibly a discount for long-term customers. Overall, I had a decent experience with the service, but ultimately decided to switch to a more affordable option.",1996,4,1,2,1,Misto,0.6046,Positivo,History
0013-MHZWF,Female,0,No,Yes,9,Yes,No,DSL,No,No,No,Yes,Yes,Yes,Month-to-month,Yes,Credit card (automatic),69.40,571.45,No,"I have been a customer with this company for 9 months now and I am very satisfied with the service. The monthly charges of $69.4 for DSL internet are reasonable and the fact that I have the convenience of automatic credit card payments makes it even better. I have had no issues with the service and have no plans to churn anytime soon. Overall, I am a happy customer and would recommend this company to others.",5378,4,4,0,0,Positivo,0.8516,Positivo,History
0013-SMEOE,Female,1,Yes,No,71,Yes,No,Fiber optic,Yes,Yes,Yes,Yes,Yes,Yes,Two year,Yes,Bank transfer (automatic),109.70,7904.25,No,"I have been a customer with this company for 71 months now and I am extremely satisfied with the service I have received. The fiber optic internet service is reliable and fast, and the automatic bank transfer payment method makes it convenient for me to pay my monthly charges of $109.7. I have had no issues with my service and have no plans to churn. I would highly recommend this company to others looking for reliable internet service.",5924,4,5,0,0,Positivo,-0.0036,Neutro,History
0014-BMAQU,Male,0,Yes,No,63,Yes,Yes,Fiber optic,Yes,No,No,Yes,No,No,Two year,Yes,Credit card (automatic),84.65,5377.80,No,"I have been a customer with this company for over five years now, and I have been extremely satisfied with the service. The fiber optic internet service is reliable and fast, and I have never experienced any issues with connectivity. The automatic credit card payment method makes it convenient for me to pay my monthly charges without having to worry about missing a payment. Overall, I am very happy with the service and would highly recommend it to others.",7589,3,3,0,0,Positiv

In [0]:
%python
# ============================================================
# Exportação da tabela intelligence para CSV (history)
# ============================================================
# O que faz: grava a tabela terra_signal_history_intelligence_Adri em um único arquivo CSV no volume, sem passar pela exibição do notebook.
# Origem:    terra_signal_history_intelligence_Adri
# Saída:     pasta saida_csv (volume bases, schema bases) com o arquivo part-...csv, com cabeçalho.
# Regras:    coalesce(1) gera um único arquivo. O download pela tela trunca o resultado; esta exportação entrega a base completa.
# ============================================================
(spark.table("hackaton_databricks.tabelas_hackaton.terra_signal_history_intelligence_Adri")
   .coalesce(1).write.mode("overwrite")
   .option("header", True)
   .csv("/Volumes/hackaton_databricks/bases/bases/saida_csv"))

CÓDIGO PARA INFERENCE TABELA 4

In [0]:
%python
# ============================================================
# Sentimento global e VADER por cliente (inference)
# ============================================================
# O que faz: calcula, por cliente, as contagens de sentimento por aspecto, o SentimentoGlobal e o sentimento do texto do feedback (VADER).
# Origem:    inference_tipada e feedback_consolidado_inference
# Saída:     DataFrame res (customerID, CustomerFeedback, QtdAspectosAvaliados, QtdPositivos, QtdNegativos, QtdMistos, SentimentoGlobal, SentimentScore, SentimentoVADER), sem criar tabela.
# Regras:    mesma lógica da history. QtdAspectosAvaliados não conta Sem avaliacao. QtdPositivos e QtdNegativos somam menções. QtdMistos é 1 se algum aspecto tem menção mista. SentimentoGlobal é Misto se há aspecto Misto ou Positivo e Negativo juntos; senão Positivo, Negativo ou Sem avaliacao. SentimentScore é o compound do VADER (vaderSentiment) sobre o texto completo, com 4 casas; SentimentoVADER é Positivo (>= 0,05), Negativo (<= -0,05) ou Neutro.
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf
import pandas as pd

@pandas_udf("double")
def vader(s: pd.Series) -> pd.Series:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    a = SentimentIntensityAnalyzer()
    return s.map(lambda t: float(a.polarity_scores(t)["compound"]) if t else None)

base = spark.sql("""
SELECT
  t.customerID,
  t.CustomerFeedback,
  a.QtdAspectosAvaliados, a.QtdPositivos, a.QtdNegativos, a.QtdMistos, a.SentimentoGlobal
FROM hackaton_databricks.tabelas_hackaton.inference_tipada t
JOIN (
  SELECT
    customerID,
    SUM(CASE WHEN Sentimento_Consolidado <> 'Sem avaliacao' THEN 1 ELSE 0 END) AS QtdAspectosAvaliados,
    SUM(Qtd_Positivas) AS QtdPositivos,
    SUM(Qtd_Negativas) AS QtdNegativos,
    MAX(Qtd_Mistas) AS QtdMistos,
    CASE
      WHEN SUM(CASE WHEN Sentimento_Consolidado = 'Misto' THEN 1 ELSE 0 END) > 0 THEN 'Misto'
      WHEN SUM(CASE WHEN Sentimento_Consolidado = 'Positivo' THEN 1 ELSE 0 END) > 0
       AND SUM(CASE WHEN Sentimento_Consolidado = 'Negativo' THEN 1 ELSE 0 END) > 0 THEN 'Misto'
      WHEN SUM(CASE WHEN Sentimento_Consolidado = 'Positivo' THEN 1 ELSE 0 END) > 0 THEN 'Positivo'
      WHEN SUM(CASE WHEN Sentimento_Consolidado = 'Negativo' THEN 1 ELSE 0 END) > 0 THEN 'Negativo'
      ELSE 'Sem avaliacao'
    END AS SentimentoGlobal
  FROM hackaton_databricks.tabelas_hackaton.feedback_consolidado_inference
  GROUP BY customerID
) a ON t.customerID = a.customerID
""")

res = (base
       .withColumn("SentimentScore", F.round(vader("CustomerFeedback"), 4))
       .withColumn("SentimentoVADER",
                   F.when(F.col("SentimentScore") >= 0.05, "Positivo")
                    .when(F.col("SentimentScore") <= -0.05, "Negativo")
                    .otherwise("Neutro")))

print("linhas:", res.count(), "| clientes distintos:", res.select("customerID").distinct().count())

res.filter(F.col("customerID") == "0239-OXEXL") \
   .drop("CustomerFeedback") \
   .show(truncate=False)

linhas: 1409 | clientes distintos: 1409
+----------+--------------------+------------+------------+---------+----------------+--------------+---------------+
|customerID|QtdAspectosAvaliados|QtdPositivos|QtdNegativos|QtdMistos|SentimentoGlobal|SentimentScore|SentimentoVADER|
+----------+--------------------+------------+------------+---------+----------------+--------------+---------------+
|0239-OXEXL|4                   |7           |0           |0        |Positivo        |0.9544        |Positivo       |
+----------+--------------------+------------+------------+---------+----------------+--------------+---------------+



In [0]:
%python
# ============================================================
# Gravação do sentimento por cliente em tabela temporária (inference)
# ============================================================
# O que faz: grava o DataFrame res (sentimento global e VADER por cliente) na tabela tmp_sentimento_adri, para uso nas células SQL seguintes.
# Origem:    DataFrame res, da célula anterior
# Saída:     tabela tmp_sentimento_adri (customerID, CustomerFeedback, QtdAspectosAvaliados, QtdPositivos, QtdNegativos, QtdMistos, SentimentoGlobal, SentimentScore, SentimentoVADER).
# Regras:    sobrescreve a tabela a cada execução. É intermediária: alimenta a terra_signal_customer_intelligence_Adri.
# ============================================================
res.write.mode("overwrite").saveAsTable("hackaton_databricks.tabelas_hackaton.tmp_sentimento_adri")

In [0]:
%sql
-- ============================================================
-- Aspecto principal e orientação de atendimento (inference)
-- ============================================================
-- O que faz: define, por cliente com aspecto crítico, o aspecto principal de atenção e o texto de orientação ao atendimento.
-- Origem:    feedback_consolidado_inference
-- Saída:     view temporária vw_orientacao_adri (customerID, PrincipalAspectoAtencao, OrientacaoAtendimento), só clientes com aspecto crítico.
-- Regras:    regra do colega aplicada ao nosso consolidado. São críticos os aspectos Negativos ou Mistos; Saida só conta se Negativa. Prioridade: Cobranca, Conexao, Preco, Atendimento, Suporte, Contrato, Saida e demais por último. O principal é o de maior prioridade (Saida só quando é o único crítico, foco preventivo). A orientação junta os textos de todos os críticos, sem repetição.
-- ============================================================

CREATE OR REPLACE TEMP VIEW vw_orientacao_adri AS
WITH criticos AS (
  SELECT customerID, Aspecto,
         CASE Aspecto
           WHEN 'Cobranca'    THEN 1
           WHEN 'Conexao'     THEN 2
           WHEN 'Preco'       THEN 3
           WHEN 'Atendimento' THEN 4
           WHEN 'Suporte'     THEN 5
           WHEN 'Contrato'    THEN 6
           ELSE 99 END AS ordem,
         CASE Aspecto
           WHEN 'Cobranca'    THEN 'Conferir eventuais dificuldades com faturamento e formas de pagamento.'
           WHEN 'Conexao'     THEN 'Investigar velocidade, estabilidade da conexão e possíveis problemas técnicos.'
           WHEN 'Preco'       THEN 'Verificar a percepção sobre o valor da mensalidade e consultar alternativas comerciais.'
           WHEN 'Atendimento' THEN 'Investigar experiências anteriores de atendimento e eventuais pendências.'
           WHEN 'Suporte'     THEN 'Verificar histórico de suporte e necessidade de assistência técnica.'
           WHEN 'Contrato'    THEN 'Esclarecer condições contratuais e verificar alternativas disponíveis.'
           WHEN 'Saida'       THEN 'Cliente informa ter cancelado ou saído sem indicar motivo. Identificar o que causou a saída e se há possibilidade de reversão.'
         END AS texto
  FROM hackaton_databricks.tabelas_hackaton.feedback_consolidado_inference
  WHERE (Sentimento_Consolidado IN ('Negativo', 'Misto') AND Aspecto <> 'Saida')
     OR (Aspecto = 'Saida' AND Sentimento_Consolidado = 'Negativo')
)
SELECT customerID,
       min_by(Aspecto, struct(ordem, Aspecto)) AS PrincipalAspectoAtencao,
       NULLIF(array_join(array_distinct(transform(
         filter(array_sort(collect_list(struct(ordem, Aspecto, texto))), x -> x.texto IS NOT NULL),
         x -> x.texto)), ' '), '') AS OrientacaoAtendimento
FROM criticos
GROUP BY customerID;

SELECT COUNT(*) AS clientes_com_critico FROM vw_orientacao_adri;

clientes_com_critico
636


In [0]:
%sql
-- ============================================================
-- Orientação de atendimento para todos os clientes (inference)
-- ============================================================
-- O que faz: lista todos os clientes da inference com aspecto principal e orientação, incluindo os sem aspecto crítico.
-- Origem:    vw_orientacao_adri e inference_tipada
-- Saída:     view temporária vw_orientacao_adri_completa (customerID, PrincipalAspectoAtencao, OrientacaoAtendimento), uma linha por cliente.
-- Regras:    quem não tem aspecto crítico recebe "Nao identificado" e o texto padrão de orientação. A consulta final mostra a contagem de clientes por aspecto principal.
-- ============================================================
CREATE OR REPLACE TEMP VIEW vw_orientacao_adri_completa AS
SELECT b.customerID,
       COALESCE(o.PrincipalAspectoAtencao, 'Nao identificado') AS PrincipalAspectoAtencao,
       COALESCE(o.OrientacaoAtendimento,
         'Não foram identificadas reclamações explícitas nos aspectos analisados. Confirmar a satisfação atual e perguntar sobre eventuais dificuldades.') AS OrientacaoAtendimento
FROM (SELECT DISTINCT customerID
      FROM hackaton_databricks.tabelas_hackaton.inference_tipada) b
LEFT JOIN vw_orientacao_adri o ON o.customerID = b.customerID;

SELECT PrincipalAspectoAtencao, COUNT(*) AS n
FROM vw_orientacao_adri_completa
GROUP BY PrincipalAspectoAtencao
ORDER BY n DESC;

PrincipalAspectoAtencao,n
Nao identificado,773
Cobranca,392
Conexao,149
Preco,72
Saida,21
Atendimento,1
Contrato,1


In [0]:
%sql
-- ============================================================
-- Tabela intelligence (inference)
-- ============================================================
-- O que faz: cria a terra_signal_customer_intelligence_Adri, uma linha por cliente, com as colunas de churn, sentimento e orientação de atendimento.
-- Origem:    terra_signal_customer_intelligence, tmp_sentimento_adri e vw_orientacao_adri_completa
-- Saída:     tabela terra_signal_customer_intelligence_Adri (customerID, ChurnScore, ChurnPredicted, Rank, PercentilPrioridade, FaixaPrioridade, Modelo, VersaoModelo, Threshold, QtdAspectosAvaliados, QtdPositivos, QtdNegativos, QtdMistos, SentimentoGlobal, SentimentScore, SentimentoVADER, Feedback_Limpo, OrientacaoAtendimento, PrincipalAspectoAtencao).
-- Regras:    as 8 colunas de churn são transplantadas da tabela do colega por customerID, sem cálculo. Sentimento e VADER vêm do nosso cálculo. Feedback_Limpo é o CustomerFeedback completo, sem tratamento. A consulta final exibe a tabela ordenada por ChurnScore.
-- ============================================================
CREATE OR REPLACE TABLE hackaton_databricks.tabelas_hackaton.terra_signal_customer_intelligence_Adri AS
SELECT s.customerID,
       c.ChurnScore, c.ChurnPredicted, c.Rank, c.PercentilPrioridade, c.FaixaPrioridade,
       c.Modelo, c.VersaoModelo, c.Threshold,
       s.QtdAspectosAvaliados, s.QtdPositivos, s.QtdNegativos, s.QtdMistos,
       s.SentimentoGlobal, s.SentimentScore, s.SentimentoVADER,
       s.CustomerFeedback AS Feedback_Limpo,
       o.OrientacaoAtendimento, o.PrincipalAspectoAtencao
FROM hackaton_databricks.tabelas_hackaton.tmp_sentimento_adri s
LEFT JOIN hackaton_databricks.tabelas_hackaton.terra_signal_customer_intelligence c
       ON c.customerID = s.customerID
LEFT JOIN vw_orientacao_adri_completa o
       ON o.customerID = s.customerID;

SELECT * FROM hackaton_databricks.tabelas_hackaton.terra_signal_customer_intelligence_Adri
ORDER BY ChurnScore DESC;

customerID,ChurnScore,ChurnPredicted,Rank,PercentilPrioridade,FaixaPrioridade,Modelo,VersaoModelo,Threshold,QtdAspectosAvaliados,QtdPositivos,QtdNegativos,QtdMistos,SentimentoGlobal,SentimentScore,SentimentoVADER,Feedback_Limpo,OrientacaoAtendimento,PrincipalAspectoAtencao
9300-AGZNL,"0,9043115248",Yes,1,"0,0709723208",0-5%,GradientBoosting,GB_v1,"0,5000000000",3,0,2,1,Misto,-0.8901,Negativo,"I recently switched to this fiber optic internet service provider and unfortunately I had to churn after only one month. The monthly charges were reasonable at $94.00, but I had some issues with the reliability of the connection. I was disappointed with the service and decided to cancel. Overall, I would not recommend this provider for anyone looking for a consistent and reliable internet connection.","Investigar velocidade, estabilidade da conexão e possíveis problemas técnicos. Verificar a percepção sobre o valor da mensalidade e consultar alternativas comerciais. Cliente informa ter cancelado ou saído sem indicar motivo. Identificar o que causou a saída e se há possibilidade de reversão.",Conexao
0107-YHINA,"0,8950936930",Yes,2,"0,1419446416",0-5%,GradientBoosting,GB_v1,"0,5000000000",4,4,4,0,Misto,0.3716,Positivo,"I recently signed up for fiber optic internet service with this company on a month-to-month contract. Unfortunately, after just one month, I decided to cancel due to some unexpected financial constraints. The monthly charges of $99.75 were a bit higher than I initially anticipated, and I found it difficult to keep up with the payments using electronic check. The internet service itself was reliable and fast, but ultimately I had to churn due to the cost. I would recommend this service to others who can afford the monthly charges and are comfortable with electronic check payments.",Conferir eventuais dificuldades com faturamento e formas de pagamento. Verificar a percepção sobre o valor da mensalidade e consultar alternativas comerciais. Cliente informa ter cancelado ou saído sem indicar motivo. Identificar o que causou a saída e se há possibilidade de reversão.,Cobranca
2868-MZAGQ,"0,8880729010",Yes,3,"0,2129169624",0-5%,GradientBoosting,GB_v1,"0,5000000000",4,1,4,1,Misto,-0.3291,Negativo,"I recently signed up for fiber optic internet service with this company on a month-to-month contract. However, after just one month of service, I decided to cancel due to some issues with the reliability of the connection. Despite the fast speeds advertised, I experienced frequent outages and slow performance. The monthly charges of $79.25 were reasonable, but I expected better service for the price. I also had some issues with the payment method using electronic check, as it seemed to take longer to process than expected. Overall, I was disappointed with the service and decided to switch to a different provider.","Conferir eventuais dificuldades com faturamento e formas de pagamento. Investigar velocidade, estabilidade da conexão e possíveis problemas técnicos. Cliente informa ter cancelado ou saído sem indicar motivo. Identificar o que causou a saída e se há possibilidade de reversão.",Cobranca
4910-GMJOT,"0,8814425326",Yes,4,"0,2838892832",0-5%,GradientBoosting,GB_v1,"0,5000000000",5,3,3,1,Misto,-0.1022,Negativo,"I recently signed up for fiber optic internet service with this company on a month-to-month contract. Unfortunately, after only one month of service, I have decided to churn and look for another provider. The monthly charges of $94.60 were reasonable, but the service did not meet my expectations. The internet speed was inconsistent and I experienced frequent outages. Additionally, the payment method of electronic check was convenient, but I expected better reliability and customer service from a fiber optic provider. Overall, I was not satisfied with my experience and will be switching to a different provider.","Investigar velocidade, estabilidade da conexão e possíveis problemas técnicos. Verificar a percepção sobre o valor da

In [0]:
%sql
-- ============================================================
-- Tabela terra_signal_customer_intelligence (Terra Signal, inference)
-- ============================================================
-- O que faz: carrega o arquivo CSV da Terra Signal (customer intelligence, inference) em tabela.
-- Origem:    terra_signal_customer_intelligence.csv (volume bases, schema bases)
-- Saída:     tabela terra_signal_customer_intelligence, com as colunas do CSV, todas como texto.
-- Regras:    separador do CSV ";" e cabeçalho na primeira linha. Os tipos não são inferidos. Alimenta as 8 colunas de churn da terra_signal_customer_intelligence_Adri.
-- ============================================================

CREATE OR REPLACE TABLE hackaton_databricks.tabelas_hackaton.terra_signal_customer_intelligence AS
SELECT *
FROM read_files(
  '/Volumes/hackaton_databricks/bases/bases/terra_signal_customer_intelligence.csv',
  format => 'csv',
  sep => ';',
  header => true,
  inferColumnTypes => false
);

SELECT COUNT(*) AS linhas
FROM hackaton_databricks.tabelas_hackaton.terra_signal_customer_intelligence;

SELECT *
FROM hackaton_databricks.tabelas_hackaton.terra_signal_customer_intelligence
LIMIT ALL;

customerID,ChurnScore,ChurnPredicted,Rank,PercentilPrioridade,FaixaPrioridade,Modelo,VersaoModelo,Threshold,QtdAspectosAvaliados,QtdPositivos,QtdNegativos,QtdMistos,SentimentoGlobal,SentimentScore,SentimentoVADER,Feedback_Limpo,OrientacaoAtendimento,PrincipalAspectoAtencao,_rescued_data
0239-OXEXL,"0,1801449381",No,705,"50,0354861604",Acima de 20%,GradientBoosting,GB_v1,"0,5000000000",2,2,0,0,Positivo,"0,9544000000",Positivo,I have been a loyal customer of this internet service provider for 46 months now and I have been extremely satisfied with the quality of their fiber optic internet service. The monthly charges of $84.25 are reasonable for the fast and reliable internet connection I receive.,null,null,null
"I appreciate the flexibility of the month-to-month contract as it allows me to easily make changes if needed. Despite paying by mailed check, I have never had any issues with billing or payment processing.",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"Overall, I am very happy with my experience with this internet service provider and would highly recommend them to others looking for a reliable and affordable internet service.""",Não foram identificadas reclamações explícitas nos aspectos analisados. Confirmar a satisfação atual e perguntar sobre eventuais dificuldades.,Nao identificado,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
1925-LFCZZ,"0,1798869631",No,706,"50,1064584812",Acima de 20%,GradientBoosting,GB_v1,"0,5000000000",2,2,0,0,Positivo,"0,9118000000",Positivo,"I have been a loyal customer with this company for over 4 and a half years now and I have had a positive experience overall. The DSL internet service has been reliable and consistent, and the monthly charges are reasonable at $50.05. I appreciate the convenience of the automatic bank transfer payment method, as it ensures that my bill is always paid on time without any hassle. I have not experienced any issues with my service or billing, so I have not had any reason to consider switching providers. I am satisfied with the level of service I have received and would recommend this company to others.",Não foram identificadas reclamações explícitas nos aspectos analisados. Confirmar a satisfação atual e perguntar sobre eventuais dificuldades.,Nao identificado,null
1697-NVVGY,"0,1796811090",No,707,"50,1774308020",Acima de 20%,GradientBoosting,GB_v1,"0,5000000000",2,1,0,1,Misto,"0,6115000000",Positivo,"""""""I have been a customer with this internet service provider for 19 months now and I am very satisfied with the service. The monthly charges are reasonable at $66.4 and I appreciate the convenience of automatic bank transfers for payment. The DSL internet service has been reliable and I have had no issues with downtime or slow speeds. Overall, I am happy with my experience and have no plans to churn in the near future.""""""","Investigar velocidade, estabilidade da conexão e possíveis problemas técnicos.",Conexao,null
0396-HUJBP,"0,1793540980",No,708,"50,2484031228",Acima de 20%,GradientBoosting,GB_v1,"0,5000000000",2,1,1,0,Misto,"0,9321000000",Positivo,"I have been a customer with this company for 2 months now and I am overall satisfied with the service. The monthly charges are reasonable at $19.3 and I appreciate the flexibility of the month-to-month contract. However, I do wish that internet service was included in my plan as it is a necessity for me. The payment method of mailing a check is a bit outdated and inconvenient for me personally, so I would prefer more modern options like online payment. Overall, I have had a positive experience with this company and would recommend it to others.",Conferir eventuais dificuldades com faturamento e formas de pagamento.,Cobranca,null
8082-GHXOP,"0,1786387914",No,709,"50,3193754436",Acima de 20%,GradientBoosting,GB_v1,"0,5000000000",2,2,0,0,Positivo,"0,9169000000",Positivo,"I have been a customer with this company for over a y